In [45]:
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

In [46]:
dataset = pd.read_csv('model_dataset.csv')
dataset.head()

,match_id,season,date,home_team,away_team,home_goals,away_goals,result,home_shots,away_shots,...,bookmaker_margin,bk_prob_home,bk_prob_draw,bk_prob_away,split,target,home_prev_cluster,home_prev_cluster_name,away_prev_cluster,away_prev_cluster_name
0,0,2016/17,2016-08-13,Burnley,Swansea,0,1,A,10,17,...,0.027389,0.405559,0.294952,0.299490,train,2,-1,Promoted / unknown,-1,Promoted / unknown
1,1,2016/17,2016-08-13,Crystal Palace,West Brom,0,1,A,14,13,...,0.025253,0.487685,0.295567,0.216749,train,2,-1,Promoted / unknown,-1,Promoted / unknown
2,2,2016/17,2016-08-13,Everton,Tottenham,1,1,D,12,13,...,0.023284,0.305389,0.287425,0.407186,train,1,-1,Promoted / unknown,-1,Promoted / unknown
3,3,2016/17,2016-08-13,Hull,Leicester,2,1,H,14,18,...,0.023560,0.217107,0.271384,0.511509,train,0,-1,Promoted / unknown,-1,Promoted / unknown
4,4,2016/17,2016-08-13,Man City,Sunderland,2,1,H,16,7,...,0.020513,0.783920,0.150754,0.065327,train,0,-1,Promoted / unknown,-1,Promoted / unknown


In [47]:
target_cols = dataset.columns[dataset.columns.str.contains('home_form|away_form|diff_form')]
dataset = dataset.dropna(subset=target_cols)
dataset

,match_id,season,date,home_team,away_team,home_goals,away_goals,result,home_shots,away_shots,...,bookmaker_margin,bk_prob_home,bk_prob_draw,bk_prob_away,split,target,home_prev_cluster,home_prev_cluster_name,away_prev_cluster,away_prev_cluster_name
30,30,2016/17,2016-09-10,Arsenal,Southampton,2,1,H,17,11,...,0.028884,0.619062,0.231411,0.149527,train,0,-1,Promoted / unknown,-1,Promoted / unknown
31,31,2016/17,2016-09-10,Bournemouth,West Brom,1,0,H,12,12,...,0.026444,0.475238,0.299765,0.224997,train,0,-1,Promoted / unknown,-1,Promoted / unknown
32,32,2016/17,2016-09-10,Burnley,Hull,1,1,D,13,11,...,0.024862,0.398262,0.314755,0.286983,train,1,-1,Promoted / unknown,-1,Promoted / unknown
33,33,2016/17,2016-09-10,Liverpool,Leicester,4,1,H,17,12,...,0.020053,0.576671,0.245085,0.178244,train,0,-1,Promoted / unknown,-1,Promoted / unknown
34,34,2016/17,2016-09-10,Man United,Man City,1,2,A,14,18,...,0.021978,0.411132,0.301075,0.287793,train,2,-1,Promoted / unknown,-1,Promoted / unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795,3795,2025/26,2026-05-24,Man City,Aston Villa,1,2,A,16,12,...,0.052972,0.736196,0.158282,0.105521,test,2,3,Dominant,0,Strong
3796,3796,2025/26,2026-05-24,Nott'm Forest,Bournemouth,1,1,D,15,17,...,0.057502,0.286553,0.252167,0.461280,test,1,2,Mid-table,0,Strong
3797,3797,2025/26,2026-05-24,Sunderland,Chelsea,2,1,H,20,10,...,0.055556,0.263158,0.263158,0.473684,test,0,-1,Promoted / unknown,0,Strong
3798,3798,2025/26,2026-05-24,Tottenham,Everton,1,0,H,21,8,...,0.053385,0.497027,0.253152,0.249821,test,0,2,Mid-table,2,Mid-table


In [48]:
train_dataset = dataset[dataset['split']=='train']
test_dataset = dataset[dataset['split']=='test']
valid_dataset = dataset[dataset['split']=='valid']

In [ ]:
train_dataset.count()
test_dataset.count()
valid_dataset.count()

In [49]:
feature_cols = [
    'elo_diff',
    'diff_form_goals_for',
    'diff_form_goals_against',
    'diff_form_sot_for',
    'diff_form_sot_against',
    'diff_form_points',
]

X_train = train_dataset[feature_cols]
y_train = train_dataset['target']

X_valid = valid_dataset[feature_cols]
y_valid = valid_dataset['target']

X_test = test_dataset[feature_cols]
y_test = test_dataset['target']

bk_cols = ['bk_prob_home',
           'bk_prob_draw',
           'bk_prob_away',
]

bk_train = train_dataset[bk_cols]
bk_valid = valid_dataset[bk_cols]
bk_test = test_dataset[bk_cols]

In [50]:
print(X_train.shape, X_valid.shape, X_test.shape)
print(X_train.isna().sum().sum(), X_valid.isna().sum().sum(), X_test.isna().sum().sum())

(2595, 6) (377, 6) (757, 6)
0 0 0


In [9]:
dataset

,match_id,season,date,home_team,away_team,home_goals,away_goals,result,home_shots,away_shots,...,bookmaker_margin,bk_prob_home,bk_prob_draw,bk_prob_away,split,target,home_prev_cluster,home_prev_cluster_name,away_prev_cluster,away_prev_cluster_name
30,30,2016/17,2016-09-10,Arsenal,Southampton,2,1,H,17,11,...,0.028884,0.619062,0.231411,0.149527,train,0,-1,Promoted / unknown,-1,Promoted / unknown
31,31,2016/17,2016-09-10,Bournemouth,West Brom,1,0,H,12,12,...,0.026444,0.475238,0.299765,0.224997,train,0,-1,Promoted / unknown,-1,Promoted / unknown
32,32,2016/17,2016-09-10,Burnley,Hull,1,1,D,13,11,...,0.024862,0.398262,0.314755,0.286983,train,1,-1,Promoted / unknown,-1,Promoted / unknown
33,33,2016/17,2016-09-10,Liverpool,Leicester,4,1,H,17,12,...,0.020053,0.576671,0.245085,0.178244,train,0,-1,Promoted / unknown,-1,Promoted / unknown
34,34,2016/17,2016-09-10,Man United,Man City,1,2,A,14,18,...,0.021978,0.411132,0.301075,0.287793,train,2,-1,Promoted / unknown,-1,Promoted / unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795,3795,2025/26,2026-05-24,Man City,Aston Villa,1,2,A,16,12,...,0.052972,0.736196,0.158282,0.105521,test,2,3,Dominant,0,Strong
3796,3796,2025/26,2026-05-24,Nott'm Forest,Bournemouth,1,1,D,15,17,...,0.057502,0.286553,0.252167,0.461280,test,1,2,Mid-table,0,Strong
3797,3797,2025/26,2026-05-24,Sunderland,Chelsea,2,1,H,20,10,...,0.055556,0.263158,0.263158,0.473684,test,0,-1,Promoted / unknown,0,Strong
3798,3798,2025/26,2026-05-24,Tottenham,Everton,1,0,H,21,8,...,0.053385,0.497027,0.253152,0.249821,test,0,2,Mid-table,2,Mid-table


In [51]:
cluster_dummies = pd.get_dummies(
    dataset[['home_prev_cluster_name', 'away_prev_cluster_name']],
    dtype=int
)
dataset_B = pd.concat([dataset, cluster_dummies], axis=1)
cols_A = feature_cols
cluster_cols = list(cluster_dummies.columns)
cols_B = cols_A + cluster_cols

train_B = dataset_B[dataset_B['split'] == 'train']
valid_B = dataset_B[dataset_B['split'] == 'valid']
test_B  = dataset_B[dataset_B['split'] == 'test']

X_train_B = train_B[cols_B]
X_valid_B = valid_B[cols_B]
X_test_B  = test_B[cols_B]

print(X_train_B.shape, X_valid_B.shape, X_test_B.shape)
print(X_train_B.isna().sum().sum())
print(cluster_cols)
X_train_B[cluster_cols].sum(axis=1).value_counts()

(2595, 16) (377, 16) (757, 16)
0
['home_prev_cluster_name_Dominant', 'home_prev_cluster_name_Mid-table', 'home_prev_cluster_name_Promoted / unknown', 'home_prev_cluster_name_Strong', 'home_prev_cluster_name_Struggling', 'away_prev_cluster_name_Dominant', 'away_prev_cluster_name_Mid-table', 'away_prev_cluster_name_Promoted / unknown', 'away_prev_cluster_name_Strong', 'away_prev_cluster_name_Struggling']


2    2595
Name: count, dtype: int64

In [52]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

In [55]:
model = LogisticRegression(C=1, max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

valid_probs = model.predict_proba(X_valid_s)
valid_preds = model.predict(X_valid_s)

ll_model = log_loss(y_valid, valid_probs)
acc_score = accuracy_score(y_valid, valid_preds)

ll_model_bk = log_loss(y_valid, bk_valid)
print("Model log loss =", round(ll_model, 4), "accuracy =", round(acc_score, 3))
print("Bk log loss =", round(ll_model_bk, 4))

Model log loss = 0.9253 accuracy = 0.578
Bk log loss = 0.9132
